# Building Your First Vector Search System

Time to apply what you’ve learned. You’ll create a complete, working vector search system from scratch.

# What You’ll Build
A working search system with:

- One collection with 4-dimensional vectors and Cosine distance
- 5–10 points with hand-crafted vectors and meaningful payloads
- Basic similarity search to find nearest neighbors
- Filtered search combining similarity with payload conditions

*Before creating data, decide what each of the four dimensions in your vectors will represent. This is the creative part of vector search!*



### What does each dimension represent?

We will do **16 personalities** representation using the 4 vectors. Each vector will have its own dimension

1. Abstract Thinking (float 0-1) 
2. Emotional awareness (float 0-1)
3. Risk tolerance (float 0-1)
4. Structure needness (float 0-1) 

#### Step 1: Initialize Client


In [3]:
import os
from qdrant_client import QdrantClient, models


URL = os.getenv('QDRANT_URL')
API_KEY = os.getenv('QDRANT_API_KEY')

if URL is None or API_KEY is None:
    raise RuntimeError("Cannot run application w/o url and api_key of qdrant")

client= QdrantClient(
    url=URL,
    api_key=API_KEY
)

client

In [7]:
len(client.get_collections().collections)

2

#### 2. Create collection

In [8]:
collection_name = 'personal_personalities'

In [9]:
has_been_created = client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=4,
        distance=models.Distance.COSINE
    )
)

has_been_created

True

In [10]:
client.create_payload_index(
    collection_name=collection_name,
    field_name='position',
    field_schema=models.PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

#### Step 3: Insert Points

In [11]:
from datetime import datetime, timezone


points = [
    models.PointStruct(
        id=2,
        vector=[
            0.1,
            0.8,
            0.2,
            0.9,
        ],  # concrete thinker, emotionally aware, risk-averse, needs structure
        payload={
            "name": "Sarah Chen",
            "position": "Startup Founder",  # MISALIGNMENT: Startups need risk-taking & flexibility
            "test_last_updated": datetime.now(timezone.utc).isoformat(),
        },
    ),
    models.PointStruct(
        id=3,
        vector=[
            0.8,
            0.3,
            0.7,
            0.3,
        ],  # abstract thinker, low emotional awareness, risk-tolerant, flexible
        payload={
            "name": "Marcus Thompson",
            "position": "Compliance Officer",  # MISALIGNMENT: Compliance needs structure & caution
            "test_last_updated": datetime.now(timezone.utc).isoformat(),
        },
    ),
    models.PointStruct(
        id=4,
        vector=[
            0.3,
            0.9,
            0.4,
            0.7,
        ],  # concrete thinker, high emotional awareness, moderate risk, structured
        payload={
            "name": "Elena Rodriguez",
            "position": "HR Manager",  # ALIGNED: Perfect fit for HR role
            "test_last_updated": datetime.now(timezone.utc).isoformat(),
        },
    ),
    models.PointStruct(
        id=5,
        vector=[
            0.9,
            0.2,
            0.8,
            0.2,
        ],  # abstract thinker, low emotional awareness, risk-tolerant, flexible
        payload={
            "name": "David Kim",
            "position": "Research Scientist",  # ALIGNED: Good fit for innovative research
            "test_last_updated": datetime.now(timezone.utc).isoformat(),
        },
    ),
    models.PointStruct(
        id=6,
        vector=[
            0.2,
            0.7,
            0.1,
            0.8,
        ],  # concrete thinker, emotionally aware, cautious, structured
        payload={
            "name": "Priya Patel",
            "position": "Data Scientist",  # MISALIGNMENT: DS often needs abstract thinking
            "test_last_updated": datetime.now(timezone.utc).isoformat(),
        },
    ),
    models.PointStruct(
        id=7,
        vector=[
            0.7,
            0.4,
            0.9,
            0.4,
        ],  # abstract thinker, moderate emotional awareness, high risk, flexible
        payload={
            "name": "James Wilson",
            "position": "Venture Capitalist",  # ALIGNED: Perfect for VC role
            "test_last_updated": datetime.now(timezone.utc).isoformat(),
        },
    ),
    models.PointStruct(
        id=8,
        vector=[
            0.4,
            0.6,
            0.2,
            0.9,
        ],  # moderate abstract, emotionally aware, cautious, highly structured
        payload={
            "name": "Lisa Zhang",
            "position": "Creative Director",  # MISALIGNMENT: Creative roles need flexibility
            "test_last_updated": datetime.now(timezone.utc).isoformat(),
        },
    ),
    models.PointStruct(
        id=9,
        vector=[
            0.1,
            0.8,
            0.3,
            0.6,
        ],  # concrete thinker, high emotional awareness, cautious, moderately structured
        payload={
            "name": "Robert Anderson",
            "position": "Customer Success Manager",  # ALIGNED: Good fit for customer-facing role
            "test_last_updated": datetime.now(timezone.utc).isoformat(),
        },
    ),
    models.PointStruct(
        id=10,
        vector=[
            0.8,
            0.2,
            0.7,
            0.1,
        ],  # abstract thinker, low emotional awareness, risk-tolerant, very flexible
        payload={
            "name": "Fatima Al-Hassan",
            "position": "Quality Assurance Lead",  # MISALIGNMENT: QA needs structure & caution
            "test_last_updated": datetime.now(timezone.utc).isoformat(),
        },
    ),
    models.PointStruct(
        id=11,
        vector=[
            0.5,
            0.6,
            0.4,
            0.7,
        ],  # balanced abstract thinking, good emotional awareness, moderate risk, structured
        payload={
            "name": "Michael O'Brien",
            "position": "Project Manager",  # ALIGNED: Balanced profile fits PM role well
            "test_last_updated": datetime.now(timezone.utc).isoformat(),
        },
    ),
]

In [12]:
result = client.upsert(collection_name=collection_name, points=points)

result

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

### Step 4: Test searches

In [17]:
# Define a query for the search 

query_vector = [0.04, 0.5, 1, 0.1] # I need really risky guy with some emotional awareness



basic_results = client.query_points(
    collection_name,
    query=query_vector,
    limit=2
)

basic_results.points[:2]

[ScoredPoint(id=7, version=3, score=0.81700456, payload={'name': 'James Wilson', 'position': 'Venture Capitalist', 'test_last_updated': '2026-06-02T13:03:49.746591+00:00'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=3, score=0.70941174, payload={'name': 'Marcus Thompson', 'position': 'Compliance Officer', 'test_last_updated': '2026-06-02T13:03:49.746568+00:00'}, vector=None, shard_key=None, order_value=None)]

In [20]:
filtered_results = client.query_points(
    collection_name,
    query=query_vector,
    query_filter=models.Filter(
        must=[models.FieldCondition(key="position", match=models.MatchValue(value="Creative Director"))]
    )
)

filtered_results

QueryResponse(points=[ScoredPoint(id=8, version=3, score=0.4609475, payload={'name': 'Lisa Zhang', 'position': 'Creative Director', 'test_last_updated': '2026-06-02T13:03:49.746594+00:00'}, vector=None, shard_key=None, order_value=None)])

In [21]:
position_ideal_vectors = {
    "Software Developer": [0.8, 0.3, 0.5, 0.6],  # Abstract, low emotion, moderate risk, some structure
    "Startup Founder": [0.6, 0.6, 0.9, 0.1],     # Abstract, emotional awareness, high risk, flexible
    "Compliance Officer": [0.1, 0.4, 0.1, 0.9],  # Concrete, some emotion, very cautious, highly structured
    "HR Manager": [0.4, 0.9, 0.3, 0.7],          # Some abstract, high emotion, cautious, structured
    "Research Scientist": [0.9, 0.2, 0.7, 0.3],  # Very abstract, low emotion, risk-taking, flexible
    "Data Scientist": [0.8, 0.3, 0.6, 0.5],      # Abstract, low emotion, some risk, balanced structure
    "Venture Capitalist": [0.7, 0.5, 0.9, 0.4],  # Abstract, moderate emotion, very high risk, flexible
    "Creative Director": [0.8, 0.7, 0.7, 0.2],   # Abstract, emotional, risk-taking, very flexible
    "Customer Success Manager": [0.3, 0.9, 0.3, 0.6], # Concrete, very high emotion, cautious, some structure
    "Quality Assurance Lead": [0.4, 0.3, 0.1, 0.9],   # Some abstract, low emotion, very cautious, highly structured
    "Project Manager": [0.5, 0.7, 0.4, 0.8]      # Balanced abstract, good emotion, moderate risk, structured
}

In [33]:
from dataclasses import dataclass
from typing import Any

from qdrant_client.models import QueryRequest


@dataclass
class OutlierResult:
    person_name: str
    current_position: str
    suggested_position: str
    similarity_score: float
    person_id: str


outliers = []

queries = [
    QueryRequest(
        query=vector,
        filter=models.Filter(
            must_not=[
                models.FieldCondition(
                    key='position',
                    match=models.MatchValue(value=position)
                )
            ]
        ),
        limit=1,
        with_payload=True
    ) for position, vector in position_ideal_vectors.items()
]



results = client.query_batch_points(
    collection_name=collection_name,
    requests=queries
)

assert len(results) == len(position_ideal_vectors)

In [35]:
outliers: list[OutlierResult] = []

for idx, (position, result) in enumerate(zip(position_ideal_vectors, results)):
    if result.points and result.points[0].score > 0.7:
        best_match = result.points[0]

        if best_match.payload is None:
            raise ValueError("Should have gotten payload from response")

        outliers.append(OutlierResult(
            person_name=best_match.payload['name'],
            current_position=best_match.payload['position'],
            suggested_position=position,
            similarity_score=best_match.score,
            person_id=str(best_match.id)
        ))

outliers

[OutlierResult(person_name='Marcus Thompson', current_position='Compliance Officer', suggested_position='Software Developer', similarity_score=0.9510043, person_id='3'),
 OutlierResult(person_name='James Wilson', current_position='Venture Capitalist', suggested_position='Startup Founder', similarity_score=0.9560026, person_id='7'),
 OutlierResult(person_name='Lisa Zhang', current_position='Creative Director', suggested_position='Compliance Officer', similarity_score=0.9531145, person_id='8'),
 OutlierResult(person_name='Robert Anderson', current_position='Customer Success Manager', suggested_position='HR Manager', similarity_score=0.97261626, person_id='9'),
 OutlierResult(person_name='Marcus Thompson', current_position='Compliance Officer', suggested_position='Research Scientist', similarity_score=0.9936541, person_id='3'),
 OutlierResult(person_name='Marcus Thompson', current_position='Compliance Officer', suggested_position='Data Scientist', similarity_score=0.9811949, person_id='3'

Day 0] Building Your First Vector Search System

  High-Level Summary
  - Domain: "I built a vector search for personality-based role alignment using 16 personalities framework"
  - Key Finding: "The vectors revealed significant role misalignments - many employees have personalities better suited for different positions"

  Project-Specific Details
  - Vector meaning: d1=Abstract Thinking (0-1), d2=Emotional Awareness (0-1), d3=Risk Tolerance (0-1), d4=Structure Needness (0-1)
  - Collection: personal_personalities (Cosine), points: 10
  - Query vectors: Position ideal vectors (e.g., Software Developer: [0.8, 0.3, 0.5, 0.6])
  - Top misalignment findings:
    a. Marcus Thompson (Compliance Officer) → 0.951 match with Software Developer
    b. Marcus Thompson (Compliance Officer) → 0.981 match with Data Scientist
    c. Lisa Zhang (Creative Director) → 0.966 match with Quality Assurance Lead
  - Filter example: position="Creative Director"
  - Filtered result: Found Lisa Zhang when searching for Creative Director

  Why these matched
  - Marcus Thompson's vector [0.8, 0.3, 0.7, 0.3] shows high abstract thinking and low structure need - opposite of compliance requirements but perfect for technical roles
  - Lisa Zhang's [0.4, 0.6, 0.2, 0.9] shows high structure need and caution - misaligned with creative freedom but ideal for QA

  Surprise
  - While most data was generated and hardcoded, one surprise was discovering how strongly some people matched roles completely different from their assigned positions - revealing
  potential talent misallocation

  Next step
  - Create a group recommendation engine for assembling the "perfect team" for a new startup venture, optimizing for complementary personalities and collective capabilities

  Technical Insight
  - The most effective approach would be filtered batch search using query_batch_points with negative filters (must_not match current position) - allowing single API call to find all
  misalignments while excluding already-correct matches